## Step 0. 패키지 설치

In [ ]:
# uv 사용 환경이라면:
# uv add langgraph langchain-openai wikipedia typing_extensions

# pip 사용 환경이라면:
# pip install -U langgraph langchain-openai wikipedia typing_extensions

## Step 1. Imports 및 API 키 설정

In [ ]:
import os
import operator
from typing import List, Literal, Annotated
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI

import wikipedia
wikipedia.set_lang("ko")

print("✅ 모든 패키지 로드 완료")

In [ ]:
# OPENAI_API_KEY 설정
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

# LLM 초기화 (API 키가 없으면 템플릿 모드로 동작)
USE_LLM = bool(os.environ.get("OPENAI_API_KEY"))

if USE_LLM:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
    print("✅ OpenAI LLM 연결 완료")
else:
    llm = None
    print("⚠️  API 키 없음 — 템플릿 모드로 실행 (OPENAI_API_KEY 설정 시 LLM 활성화)")

## Step 2. Tool 정의 (3개)

`@tool` 데코레이터로 LangChain 도구를 정의합니다.

| Tool | 역할 |
|------|------|
| `search_wikipedia` | 주제 관련 Wikipedia 문서 검색 |
| `assess_difficulty` | 학습 문제 난이도 자동 평가 |
| `generate_quiz` | 난이도별 퀴즈 문제 생성 |

In [ ]:
# ─── Tool 1: Wikipedia 검색 ───────────────────────────────────────────────────
@tool
def search_wikipedia(query: str) -> str:
    """Wikipedia에서 주제 관련 교육 자료를 검색합니다."""
    try:
        results = wikipedia.search(query, results=2)
        if not results:
            return "관련 Wikipedia 문서를 찾지 못했습니다."
        page = wikipedia.page(results[0])
        # 처음 600자만 반환
        return f"[Wikipedia: {page.title}]\n{page.summary[:600]}"
    except wikipedia.exceptions.DisambiguationError as e:
        # 동음이의어 페이지인 경우 첫 번째 옵션 시도
        try:
            page = wikipedia.page(e.options[0])
            return f"[Wikipedia: {page.title}]\n{page.summary[:600]}"
        except Exception:
            return "Wikipedia 검색 결과가 모호합니다. 더 구체적인 키워드를 사용해보세요."
    except Exception as e:
        return f"검색 중 오류 발생: {type(e).__name__}"


# ─── Tool 2: 난이도 평가 ──────────────────────────────────────────────────────
@tool
def assess_difficulty(problem: str) -> str:
    """
    학습 문제의 난이도를 평가합니다.
    반환값: 'beginner' | 'intermediate' | 'advanced'
    """
    advanced_keywords = [
        "함수", "알고리즘", "미적분", "적분", "미분",
        "물리", "화학", "유기", "코딩", "프로그래밍",
        "재귀", "클래스", "객체", "포인터", "자료구조",
        "수학", "통계", "확률", "행렬", "벡터",
    ]
    intermediate_keywords = [
        "공부", "시험", "집중", "암기", "이해",
        "독해", "작문", "문법", "영어", "회화",
    ]

    if any(kw in problem for kw in advanced_keywords):
        return "advanced"
    elif any(kw in problem for kw in intermediate_keywords):
        return "intermediate"
    return "beginner"


# ─── Tool 3: 퀴즈 생성 ───────────────────────────────────────────────────────
@tool
def generate_quiz(topic: str, difficulty: str) -> List[str]:
    """주제와 난이도에 맞는 퀴즈 문제 목록을 생성합니다."""
    templates: dict = {
        "beginner": [
            f"'{topic}'를 모르는 친구에게 한 문장으로 설명해보세요.",
            f"'{topic}'와 관련된 쉬운 예시를 하나 만들어보세요.",
            f"'{topic}'를 배우면 어디에 활용할 수 있을까요?",
        ],
        "intermediate": [
            f"'{topic}'의 핵심 원리를 설명하고 실생활 예시를 드세요.",
            f"'{topic}'와 관련된 유사 개념 2가지를 비교해보세요.",
            f"'{topic}'을 활용해 직접 문제를 하나 만들고 풀어보세요.",
        ],
        "advanced": [
            f"'{topic}'의 심화 개념과 예외 상황을 설명하세요.",
            f"'{topic}'을 활용한 복잡한 시나리오를 설계하고 분석하세요.",
            f"'{topic}'을 완전히 마스터했다면 어떤 문제를 풀 수 있을까요?",
        ],
    }
    return templates.get(difficulty, templates["intermediate"])


print("✅ Tool 3개 정의 완료:", [search_wikipedia.name, assess_difficulty.name, generate_quiz.name])

## Step 3. State 정의 (확장)

기존 State에 다음 필드를 추가했습니다.

| 필드 | 타입 | 용도 |
|------|------|------|
| `difficulty_level` | str | Tool 평가 결과 저장 |
| `needs_search` | bool | **Conditional Edge** 분기 트리거 |
| `search_results` | str | Wikipedia Tool 결과 저장 |
| `conversation_history` | List[dict] | **메모리** — 세션 이력 누적 |

In [ ]:
class LearningProblemState(TypedDict):
    # 기존 필드
    user_problem: str
    analysis: str
    learning_goal: str
    study_plan: List[str]
    practice_tasks: List[str]
    feedback: str

    # 신규 필드 ↓
    difficulty_level: str          # assess_difficulty Tool 결과
    needs_search: bool             # Conditional Edge 분기점
    search_results: str            # web_searcher 노드 결과
    conversation_history: Annotated[List[dict], operator.add]  # 메모리 누적

print("✅ State 정의 완료")

## Step 4. Conditional Edge 라우팅 함수

**Conditional Edge**는 특정 노드 실행 후 다음에 어느 노드로 갈지를 동적으로 결정합니다.

```
problem_analyzer 실행 후
    → needs_search=True  : 'web_searcher' 로 이동
    → needs_search=False : 'goal_setter'  로 이동
```

In [ ]:
def route_after_analysis(state: LearningProblemState) -> Literal["web_searcher", "goal_setter"]:
    """
    Conditional Edge 라우팅 함수.
    needs_search 값을 읽어 다음 노드를 결정합니다.
    """
    if state.get("needs_search", False):
        print("  🔍 [라우팅] 기술적 주제 감지 → web_searcher 경로")
        return "web_searcher"
    else:
        print("  💭 [라우팅] 일반 학습 문제 → goal_setter 경로")
        return "goal_setter"

print("✅ Conditional Edge 함수 정의 완료")

## Step 5. 노드 구현 (6개)

### 노드 목록

| 노드 | 역할 | 사용 Tool/LLM |
|------|------|---------------|
| `problem_analyzer` | 문제 분석 + 경로 결정 | `assess_difficulty` Tool |
| `web_searcher` | Wikipedia 검색 **(조건부 실행)** | `search_wikipedia` Tool |
| `goal_setter` | 학습 목표 설정 | LLM (없으면 템플릿) |
| `solution_planner` | 단계별 계획 생성 | LLM (없으면 템플릿) |
| `quiz_generator` | 퀴즈 문제 생성 | `generate_quiz` Tool |
| `feedback_coach` | 최종 피드백 | LLM (없으면 템플릿) |

In [ ]:
def _invoke_llm(system: str, human: str, fallback: str) -> str:
    """LLM 호출 헬퍼. API 키 없으면 fallback 반환."""
    if llm is None:
        return fallback
    try:
        response = llm.invoke([
            SystemMessage(content=system),
            HumanMessage(content=human),
        ])
        return response.content
    except Exception as e:
        return f"[LLM 오류: {e}]\n\n{fallback}"


# ─── 노드 1: problem_analyzer ────────────────────────────────────────────────
def problem_analyzer(state: LearningProblemState) -> dict:
    """문제 분석 + needs_search 결정 (Conditional Edge 트리거)"""
    print("\n[1] problem_analyzer 실행")
    problem = state["user_problem"]

    # Tool 1 사용: 난이도 평가
    difficulty = assess_difficulty.invoke({"problem": problem})
    print(f"  → assess_difficulty Tool: {difficulty}")

    # 기술적 주제 키워드 감지 → Conditional Edge 결정값
    technical_keywords = [
        "함수", "알고리즘", "미적분", "코딩", "프로그래밍",
        "수학", "물리", "화학", "재귀", "클래스", "자료구조",
    ]
    needs_search = any(kw in problem for kw in technical_keywords)

    # LLM 분석 (없으면 템플릿)
    analysis = _invoke_llm(
        system="당신은 교육 전문가입니다. 학습 문제를 분석하고 핵심 원인을 파악합니다.",
        human=(
            f"다음 학습 문제를 분석해주세요:\n'{problem}'\n\n"
            "핵심 문제, 가능한 원인 2-3가지를 한국어로 간결하게 정리해주세요."
        ),
        fallback=(
            f"[문제 분석]\n"
            f"사용자의 문제: '{problem}'\n"
            f"난이도: {difficulty}\n\n"
            "가능한 원인:\n"
            "- 핵심 개념에 대한 이해 부족\n"
            "- 충분한 반복 연습 부족\n"
            "- 체계적 학습 계획 없음"
        ),
    )

    return {
        "analysis": analysis,
        "difficulty_level": difficulty,
        "needs_search": needs_search,
        "conversation_history": [{"node": "problem_analyzer", "content": analysis}],
    }


# ─── 노드 2: web_searcher (조건부 실행) ─────────────────────────────────────
def web_searcher(state: LearningProblemState) -> dict:
    """Wikipedia 검색 (needs_search=True일 때만 실행되는 노드)"""
    print("[2] web_searcher 실행 (Wikipedia Tool)")
    problem = state["user_problem"]

    # Tool 1 사용: Wikipedia 검색
    search_results = search_wikipedia.invoke({"query": problem[:40]})
    print(f"  → Wikipedia 검색 완료 ({len(search_results)}자 수집)")

    return {
        "search_results": search_results,
        "conversation_history": [{"node": "web_searcher", "content": search_results[:100] + "..."}],
    }


# ─── 노드 3: goal_setter ─────────────────────────────────────────────────────
def goal_setter(state: LearningProblemState) -> dict:
    """분석 결과 + 검색 결과를 바탕으로 학습 목표 설정"""
    print("[3] goal_setter 실행")
    problem = state["user_problem"]
    search_info = state.get("search_results", "")
    search_context = f"\n\n참고 자료:\n{search_info}" if search_info else ""

    learning_goal = _invoke_llm(
        system="당신은 학습 코치입니다. 막연한 고민을 구체적이고 달성 가능한 학습 목표로 바꿉니다.",
        human=(
            f"학습 문제: '{problem}'{search_context}\n\n"
            "이 문제를 해결하기 위한 SMART(구체적, 측정가능, 달성가능, 관련성, 시한)한 "
            "학습 목표를 2주 기준으로 한국어로 설정해주세요."
        ),
        fallback=(
            f"[학습 목표]\n"
            f"'{problem}'를 해결하기 위해, 앞으로 2주 동안\n"
            "매일 20~30분씩 집중 학습하여 관련 개념을 스스로 설명하고\n"
            "기본 문제 5개 이상을 풀 수 있는 상태를 목표로 합니다."
        ),
    )

    return {
        "learning_goal": learning_goal,
        "conversation_history": [{"node": "goal_setter", "content": learning_goal}],
    }


# ─── 노드 4: solution_planner ────────────────────────────────────────────────
def solution_planner(state: LearningProblemState) -> dict:
    """단계별 학습 계획 생성"""
    print("[4] solution_planner 실행")
    problem = state["user_problem"]
    difficulty = state.get("difficulty_level", "intermediate")
    goal = state.get("learning_goal", "")

    plan_text = _invoke_llm(
        system="당신은 학습 플래너입니다. 실행 가능한 단계별 학습 계획을 만듭니다.",
        human=(
            f"문제: '{problem}'\n난이도: {difficulty}\n목표: {goal}\n\n"
            "6단계 학습 계획을 번호 목록 형식으로 한국어로 작성해주세요."
        ),
        fallback=None,
    )

    if plan_text:
        # LLM 응답을 줄 단위로 파싱
        lines = [l.strip() for l in plan_text.strip().split("\n") if l.strip()]
        study_plan = lines[:6] if len(lines) >= 3 else [
            "1단계: 현재 어려운 부분을 한 문장으로 정리한다.",
            "2단계: 원인을 지식/연습/집중/피드백 부족 중 분류한다.",
            "3단계: 아주 쉬운 예시 1개로 핵심 개념을 다시 학습한다.",
            "4단계: 기본 연습 문제 3개를 풀어본다.",
            "5단계: 틀린 부분을 기록하고 다음 날 복습한다.",
            "6단계: 1주일 후 같은 유형의 문제로 성장을 확인한다.",
        ]
    else:
        study_plan = [
            "1단계: 현재 어려운 부분을 한 문장으로 정리한다.",
            "2단계: 원인을 지식/연습/집중/피드백 부족 중 분류한다.",
            "3단계: 아주 쉬운 예시 1개로 핵심 개념을 다시 학습한다.",
            "4단계: 기본 연습 문제 3개를 풀어본다.",
            "5단계: 틀린 부분을 기록하고 다음 날 복습한다.",
            "6단계: 1주일 후 같은 유형의 문제로 성장을 확인한다.",
        ]

    return {
        "study_plan": study_plan,
        "conversation_history": [{"node": "solution_planner", "content": str(study_plan)}],
    }


# ─── 노드 5: quiz_generator ──────────────────────────────────────────────────
def quiz_generator(state: LearningProblemState) -> dict:
    """generate_quiz Tool을 사용해 맞춤 퀴즈 생성"""
    print("[5] quiz_generator 실행 (generate_quiz Tool)")
    problem = state["user_problem"]
    difficulty = state.get("difficulty_level", "intermediate")

    # 핵심 주제어 추출 (첫 10글자 사용)
    topic = problem[:20].rstrip()

    # Tool 3 사용: 퀴즈 생성
    practice_tasks = generate_quiz.invoke({"topic": topic, "difficulty": difficulty})
    print(f"  → {difficulty} 난이도 퀴즈 {len(practice_tasks)}개 생성")

    return {
        "practice_tasks": practice_tasks,
        "conversation_history": [{"node": "quiz_generator", "content": str(practice_tasks)}],
    }


# ─── 노드 6: feedback_coach ──────────────────────────────────────────────────
def feedback_coach(state: LearningProblemState) -> dict:
    """전체 세션을 요약하고 최종 피드백 제공"""
    print("[6] feedback_coach 실행")
    problem = state["user_problem"]
    difficulty = state.get("difficulty_level", "intermediate")
    used_search = bool(state.get("search_results"))

    search_note = "Wikipedia 자료를 참조하여 " if used_search else ""

    feedback = _invoke_llm(
        system="당신은 격려하는 학습 멘토입니다. 학습자에게 동기를 부여하고 핵심 행동을 제안합니다.",
        human=(
            f"학습 문제: '{problem}'\n난이도: {difficulty}\n"
            f"학습 목표: {state.get('learning_goal', '')}\n\n"
            "격려의 말과 함께 오늘 당장 시작할 수 있는 첫 번째 행동 1가지를 한국어로 제안해주세요."
        ),
        fallback=(
            f"[최종 피드백]\n"
            f"{search_note}분석한 결과, 지금 '{problem}' 문제는 한 번에 해결하려 하면 "
            "부담이 커질 수 있습니다.\n\n"
            "💡 오늘의 첫 행동:\n"
            "'내가 정확히 어디서 막히는지'를 한 문장으로 적어보세요.\n"
            "그 다음 가장 쉬운 예시 하나를 직접 설명해 보는 것부터 시작하면 됩니다."
        ),
    )

    return {
        "feedback": feedback,
        "conversation_history": [{"node": "feedback_coach", "content": feedback}],
    }


print("✅ 노드 6개 정의 완료")

## Step 6. 그래프 연결 + Conditional Edge + 메모리

```python
# 핵심: add_conditional_edges 로 분기 추가
graph_builder.add_conditional_edges(
    "problem_analyzer",      # 출발 노드
    route_after_analysis,    # 라우팅 함수
    {
        "web_searcher": "web_searcher",   # 기술적 주제
        "goal_setter":  "goal_setter",    # 일반 주제
    }
)
```

In [ ]:
graph_builder = StateGraph(LearningProblemState)

# ── 노드 등록 ──────────────────────────────────────────────────────────────
graph_builder.add_node("problem_analyzer", problem_analyzer)
graph_builder.add_node("web_searcher",     web_searcher)
graph_builder.add_node("goal_setter",      goal_setter)
graph_builder.add_node("solution_planner", solution_planner)
graph_builder.add_node("quiz_generator",   quiz_generator)
graph_builder.add_node("feedback_coach",   feedback_coach)

# ── 일반 엣지 ──────────────────────────────────────────────────────────────
graph_builder.add_edge(START, "problem_analyzer")

# ── Conditional Edge (핵심) ────────────────────────────────────────────────
graph_builder.add_conditional_edges(
    "problem_analyzer",
    route_after_analysis,
    {
        "web_searcher": "web_searcher",  # 기술적 주제 경로
        "goal_setter":  "goal_setter",   # 일반 학습 경로
    }
)

# web_searcher 이후에는 goal_setter 로 합류
graph_builder.add_edge("web_searcher",     "goal_setter")
graph_builder.add_edge("goal_setter",      "solution_planner")
graph_builder.add_edge("solution_planner", "quiz_generator")
graph_builder.add_edge("quiz_generator",   "feedback_coach")
graph_builder.add_edge("feedback_coach",   END)

# ── 메모리 설정 (MemorySaver) ──────────────────────────────────────────────
memory = MemorySaver()
app = graph_builder.compile(checkpointer=memory)

print("✅ 그래프 컴파일 완료 (MemorySaver 적용)")

## Step 7. 그래프 시각화

In [ ]:
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    # Mermaid PNG 생성 불가 시 텍스트 표현
    print(app.get_graph().draw_mermaid())

## Step 8. 결과 출력 헬퍼

In [ ]:
def print_result(result: dict) -> None:
    border = "=" * 70
    print(f"\n{border}")
    print(f"📌 입력 문제: {result['user_problem']}")
    print(f"🎯 난이도: {result.get('difficulty_level', '?')}")
    print(f"🔍 웹 검색 경로: {'예 (web_searcher 실행)' if result.get('search_results') else '아니오 (건너뜀)'}")
    print(border)

    print("\n[ 문제 분석 ]")
    print(result.get("analysis", ""))

    if result.get("search_results"):
        print("\n[ Wikipedia 검색 결과 ]")
        print(result["search_results"][:300] + "...")

    print("\n[ 학습 목표 ]")
    print(result.get("learning_goal", ""))

    print("\n[ 학습 계획 ]")
    for step in result.get("study_plan", []):
        print(f"  {step}")

    print("\n[ 퀴즈 / 연습 과제 ]")
    for task in result.get("practice_tasks", []):
        print(f"  ▸ {task}")

    print("\n[ 최종 피드백 ]")
    print(result.get("feedback", ""))

    print("\n[ 메모리: 대화 이력 ]")
    for entry in result.get("conversation_history", []):
        print(f"  [{entry['node']}] {str(entry['content'])[:60]}...")

    print(border)

print("✅ 출력 헬퍼 정의 완료")

## Step 9. 테스트 1: 기술적 주제 (web_searcher 경로)

기술적 키워드(`함수`, `코딩`, `알고리즘` 등) → `needs_search=True` → **web_searcher → goal_setter** 경로

In [ ]:
initial_state_1 = {
    "user_problem": "코딩 공부를 시작했는데 함수 개념이 너무 어렵습니다.",
    "analysis": "",
    "learning_goal": "",
    "study_plan": [],
    "practice_tasks": [],
    "feedback": "",
    "difficulty_level": "",
    "needs_search": False,
    "search_results": "",
    "conversation_history": [],
}

# thread_id 로 메모리 세션 구분
config_1 = {"configurable": {"thread_id": "session-1"}}

result_1 = app.invoke(initial_state_1, config=config_1)
print_result(result_1)

## Step 10. 테스트 2: 일반 학습 고민 (goal_setter 직접 경로)

일반 키워드(`집중`, `시험`, `영어` 등) → `needs_search=False` → **goal_setter** 경로 (web_searcher 건너뜀)

In [ ]:
initial_state_2 = {
    "user_problem": "시험 공부를 해야 하는데 집중력이 너무 떨어집니다.",
    "analysis": "",
    "learning_goal": "",
    "study_plan": [],
    "practice_tasks": [],
    "feedback": "",
    "difficulty_level": "",
    "needs_search": False,
    "search_results": "",
    "conversation_history": [],
}

config_2 = {"configurable": {"thread_id": "session-2"}}

result_2 = app.invoke(initial_state_2, config=config_2)
print_result(result_2)

## Step 11. 메모리 확인 — 동일 thread_id로 상태 재조회

In [ ]:
# MemorySaver를 통해 이전 세션 상태 복원
snapshot_1 = app.get_state(config_1)
print("=== session-1 저장된 상태 ===")
print(f"문제: {snapshot_1.values['user_problem']}")
print(f"난이도: {snapshot_1.values['difficulty_level']}")
print(f"웹검색: {bool(snapshot_1.values.get('search_results'))}")
print(f"대화 이력 노드 수: {len(snapshot_1.values['conversation_history'])}개")

## Step 12. 경로 분기 비교 요약

| 입력 | needs_search | 실행 경로 |
|------|-------------|----------|
| 함수 개념이 어렵다 (기술) | `True` | `problem_analyzer → web_searcher → goal_setter → ...` |
| 집중력이 떨어진다 (일반) | `False` | `problem_analyzer → goal_setter → ...` |

In [ ]:
print("== Conditional Edge 분기 결과 비교 ==")
print()
for label, result in [("테스트1 (기술)", result_1), ("테스트2 (일반)", result_2)]:
    path = "web_searcher → goal_setter" if result.get("search_results") else "goal_setter (직접)"
    print(f"{label}")
    print(f"  문제     : {result['user_problem']}")
    print(f"  난이도   : {result.get('difficulty_level')}")
    print(f"  실행경로 : problem_analyzer → {path} → solution_planner → quiz_generator → feedback_coach")
    print()